# Crush Glint-2 — 1.4M dense on Colab T4 (v2)

Locked recipe (`research/reports/11_crush_glint2_recipe.md`):

- **Model:** `dense_1_4m` (1,406,506 params, GQA + QK-norm + tied + VR)
- **Tok:** 4096 byte-BPE
- **Mix:** FineWeb-Edu **55%** / DCLM **28%** / TinyStories **12%** / soft-QA **5%**
- **Train:** AdamW 3e-3, **wall-clock WSD**, AMP + compile, packed windows, EMA
- **Seq:** prefer **1024** (fallback 512) · ~65k tok/step · **≤4h**
- **Eval:** end-of-run BLiMP / ARC-Easy / WikiText vs Glint-2 targets
- **Backup:** Google Drive every N steps

Runtime → Change runtime type → **T4 GPU**. Then run the single cell below.


In [ ]:
# @title Crush Glint-2 v2 — one-cell T4 trainer (Drive backup + Glint eval)
# Recipe: dense_1_4m + FineWeb-Edu-heavy mix + AMP/compile + wall-clock WSD + EMA.
# See research/reports/11_crush_glint2_recipe.md.

from __future__ import annotations

import json
import math
import os
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path

# ---------------------------------------------------------------------------
# Knobs (tuned for beating Glint-2 on a free Colab T4 ~4h run)
# ---------------------------------------------------------------------------
HOURS = 4.0
DRIVE_ROOT = Path("/content/drive/MyDrive/crush_glint2_1_4m")
WORK = Path("/content/crush_glint2")
REPO_URL = "https://github.com/Enderchefcoder/minimodel-trainer.git"
REPO_BRANCH = "main"
VOCAB = 4096
# Prefer longer context (sandbox: longer windows win at equal tokens). Falls back.
SEQ_CANDIDATES = (1024, 512)
BATCH_FOR_SEQ = {1024: 16, 512: 32}
ACCUM = 4  # 16*1024*4 = 65_536 or 32*512*4 = 65_536
LR = 3e-3
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
SAVE_EVERY = 200
SOFT_KL_EVERY = 10
SOFT_KL_WEIGHT = 0.30
SOFT_KL_BATCH = 12
TOK_SAMPLE_DOCS = 12_000
STREAM_BUFFER_TOKENS = 3_000_000
# Edu-heavy for ARC/WikiText; soft-QA oversampled so answers stick; DCLM for HS.
MIX = {"fineweb-edu": 0.55, "dclm": 0.28, "tinystories": 0.12, "soft_qa": 0.05}
FINEWEB_MIN_SCORE = 2.5  # FineWeb-Edu quality score floor when the field exists
EMA_DECAY = 0.999
WARMUP_FRAC = 0.03
DECAY_START_FRAC = 0.80  # last 20% of wall clock → sqrt decay (WSD)
SEED = 1337
# Glint-2 honest targets (report 10). Beat these.
GLINT_TARGETS = {
    "params": 1_710_049,
    "arc_easy_acc": 36.8,
    "wikitext_byte_ppl": 3.18,
    "blimp_acc": 66.0,  # their 73.96 unreproduced; ~66 is the honest floor
}

# ---------------------------------------------------------------------------
# 1) Drive + deps + repo
# ---------------------------------------------------------------------------
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pyyaml",
        "tqdm",
        "requests",
        "datasets",
        "huggingface_hub",
    ]
)
try:
    import torch
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ]
    )
    import torch

WORK.mkdir(parents=True, exist_ok=True)
repo_dir = WORK / "minimodel-trainer"
if not (repo_dir / "src").exists():
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(repo_dir)]
    )
else:
    # Pick up recipe fixes if the notebook is re-run on a stale clone.
    subprocess.call(["git", "-C", str(repo_dir), "pull", "--ff-only"], stdout=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)])

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir / "research" / "experiments"))
from minimodel.architectures import build_model
from minimodel.datasets.soft_labels import (
    align_steps_to_tokenizer,
    load_soft_label_dataset,
    soft_kl_loss,
    write_plain_jsonl,
)
from minimodel.tokenization import BPETokenizer
from minimodel.training.optim import build_optimizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = device.type == "cuda"
# T4: fp16 + GradScaler is the reliable AMP path (bf16 support is weak).
amp_dtype = torch.float16 if use_cuda else torch.float32
scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)
print(
    f"device={device}  torch={torch.__version__}  "
    f"cuda={torch.cuda.get_device_name(0) if use_cuda else 'n/a'}  amp={amp_dtype}"
)

# ---------------------------------------------------------------------------
# 2) Soft-label corpus on disk (+ Drive mirror)
# ---------------------------------------------------------------------------
corpus_src = repo_dir / "research" / "data" / "corpus" / "slm_next_token_dataset.json"
corpus_dir = WORK / "corpus"
corpus_dir.mkdir(parents=True, exist_ok=True)
soft_json = corpus_dir / "slm_next_token_dataset.json"
soft_jsonl = corpus_dir / "slm_next_token_qa.jsonl"
if not soft_json.exists():
    shutil.copy2(corpus_src, soft_json)
if not soft_jsonl.exists():
    write_plain_jsonl(soft_jsonl, soft_json)
if IN_COLAB:
    (DRIVE_ROOT / "corpus").mkdir(parents=True, exist_ok=True)
    shutil.copy2(soft_json, DRIVE_ROOT / "corpus" / soft_json.name)
    shutil.copy2(soft_jsonl, DRIVE_ROOT / "corpus" / soft_jsonl.name)

soft_payload = load_soft_label_dataset(soft_json)
soft_entries = soft_payload["entries"]
print(
    f"soft-label entries={len(soft_entries)}  "
    f"steps≈{soft_payload['stats']['total_predicted_token_steps']}"
)

# ---------------------------------------------------------------------------
# 3) Streaming mixture → text iterator
# ---------------------------------------------------------------------------
from datasets import load_dataset  # heavy optional dep; installed above


def _stream(name: str):
    if name == "fineweb-edu":
        return load_dataset(
            "HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True
        )
    if name == "dclm":
        return load_dataset("HuggingFaceFW/dclm_100BT", split="train", streaming=True)
    if name == "tinystories":
        return load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    raise ValueError(name)


def soft_qa_cycle():
    while True:
        for row in soft_jsonl.open("r", encoding="utf-8"):
            yield json.loads(row)["text"]


streams = {
    "fineweb-edu": iter(_stream("fineweb-edu")),
    "dclm": iter(_stream("dclm")),
    "tinystories": iter(_stream("tinystories")),
    "soft_qa": soft_qa_cycle(),
}
names = list(MIX.keys())
weights = [MIX[n] for n in names]
assert abs(sum(weights) - 1.0) < 1e-6, MIX
rng = torch.Generator().manual_seed(SEED)


def next_doc() -> str:
    idx = int(torch.multinomial(torch.tensor(weights), 1, generator=rng).item())
    name = names[idx]
    if name == "soft_qa":
        return next(streams[name])
    while True:
        row = next(streams[name])
        text = (row.get("text") or "").strip()
        if len(text) < 64:
            continue
        if name == "fineweb-edu":
            score = row.get("score")
            if score is not None and float(score) < FINEWEB_MIN_SCORE:
                continue
        return text


# ---------------------------------------------------------------------------
# 4) Train tokenizer on a mix sample (or resume from Drive)
# ---------------------------------------------------------------------------
tok_path = WORK / "tokenizer.json"
drive_tok = DRIVE_ROOT / "tokenizer.json" if IN_COLAB else None
if drive_tok and drive_tok.exists() and not tok_path.exists():
    shutil.copy2(drive_tok, tok_path)
if tok_path.exists():
    tok = BPETokenizer.load(tok_path)
    print("loaded tokenizer", tok)
else:
    print(f"training byte-BPE vocab={VOCAB} on {TOK_SAMPLE_DOCS} docs…")
    sample = [next_doc() for _ in range(TOK_SAMPLE_DOCS)]
    sample.extend(
        json.loads(line)["text"]
        for line in soft_jsonl.read_text(encoding="utf-8").splitlines()
        if line
    )
    tok = BPETokenizer.train(sample, vocab_size=VOCAB)
    tok.save(tok_path)
    if IN_COLAB:
        shutil.copy2(tok_path, drive_tok)
    print(tok, "bytes/token≈", round(tok.compression_ratio(sample[:64]), 2))


def encode(s: str) -> list[int]:
    return tok.encode(s, add_bos=False, add_eos=False)


aligned_soft = []
for entry in soft_entries:
    aligned_soft.extend(align_steps_to_tokenizer(entry, encode))
print(f"aligned soft steps={len(aligned_soft)}")

# ---------------------------------------------------------------------------
# 5) Model + AMP shape probe + optim + wall-clock WSD
# ---------------------------------------------------------------------------
raw_model = build_model("dense_1_4m", overrides={"vocab_size": tok.vocab_size}).to(device)
n_params = raw_model.num_parameters()
assert abs(n_params - 1_406_506) < 50_000 or tok.vocab_size != 4096, n_params
print(f"model params={n_params:,}")

SEQ = SEQ_CANDIDATES[-1]
BATCH = BATCH_FOR_SEQ[SEQ]
for cand in SEQ_CANDIDATES:
    b = BATCH_FOR_SEQ[cand]
    try:
        raw_model.train()
        x = torch.randint(0, tok.vocab_size, (b, cand), device=device)
        y = torch.randint(0, tok.vocab_size, (b, cand), device=device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_cuda):
            out = raw_model.forward_with_loss(x, y)
        out.loss.backward()
        raw_model.zero_grad(set_to_none=True)
        if use_cuda:
            torch.cuda.empty_cache()
        SEQ, BATCH = cand, b
        print(f"shape OK: seq={SEQ} batch={BATCH} accum={ACCUM}")
        break
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        raw_model.zero_grad(set_to_none=True)
        if use_cuda:
            torch.cuda.empty_cache()
        print(f"OOM at seq={cand} batch={b}; trying shorter…")

tokens_per_step = BATCH * SEQ * ACCUM
print(f"tokens/step={tokens_per_step:,}")

# torch.compile after the probe so Inductor sees the final shape.
model = raw_model
if use_cuda:
    try:
        model = torch.compile(raw_model, mode="default")
        print("torch.compile enabled")
    except Exception as exc:  # noqa: BLE001 — Colab Triton gaps are common
        print(f"torch.compile skipped: {exc}")
        model = raw_model

optimizer = build_optimizer(
    raw_model, "adamw", lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)
)

# Shadow EMA for eval / export (vision trainer pattern, local to this notebook).
ema_state = {k: v.detach().float().clone() for k, v in raw_model.state_dict().items()}


def ema_update() -> None:
    with torch.no_grad():
        for k, v in raw_model.state_dict().items():
            if not v.is_floating_point():
                ema_state[k] = v.detach().clone()
                continue
            ema_state[k].mul_(EMA_DECAY).add_(v.detach().float(), alpha=1.0 - EMA_DECAY)


def load_ema_into(target) -> dict:
    backup = {k: v.detach().clone() for k, v in target.state_dict().items()}
    casted = {
        k: v.to(dtype=backup[k].dtype) if torch.is_floating_point(backup[k]) else v
        for k, v in ema_state.items()
    }
    target.load_state_dict(casted)
    return backup


def wall_clock_lr(progress: float) -> float:
    """WSD on wall-clock fraction in [0, 1]: warmup → plateau → sqrt decay."""
    progress = min(max(progress, 0.0), 1.0)
    if progress < WARMUP_FRAC:
        return LR * (progress / max(WARMUP_FRAC, 1e-8))
    if progress < DECAY_START_FRAC:
        return LR
    t = (progress - DECAY_START_FRAC) / max(1.0 - DECAY_START_FRAC, 1e-8)
    return LR * max(0.05, math.sqrt(max(0.0, 1.0 - t)))


ckpt_dir = WORK / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt = DRIVE_ROOT / "checkpoints" if IN_COLAB else None
if drive_ckpt:
    drive_ckpt.mkdir(parents=True, exist_ok=True)

start_step = 0
tokens_seen = 0
best_loss = float("inf")
resume = ckpt_dir / "latest.pt"
if drive_ckpt and (drive_ckpt / "latest.pt").exists() and not resume.exists():
    shutil.copy2(drive_ckpt / "latest.pt", resume)
if resume.exists():
    blob = torch.load(resume, map_location=device, weights_only=False)
    raw_model.load_state_dict(blob["model"])
    optimizer.load_state_dict(blob["optimizer"])
    start_step = int(blob["step"])
    tokens_seen = int(blob["tokens_seen"])
    best_loss = float(blob.get("best_loss", best_loss))
    if blob.get("ema"):
        for k, v in blob["ema"].items():
            if k in ema_state:
                ema_state[k] = v.to(device=ema_state[k].device, dtype=torch.float32)
    print(f"resumed step={start_step} tokens={tokens_seen:,} best_loss={best_loss:.4f}")


def save_ckpt(step: int, loss: float, *, tag: str = "latest") -> None:
    payload = {
        "step": step,
        "tokens_seen": tokens_seen,
        "loss": loss,
        "best_loss": best_loss,
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "ema": {k: v.cpu() for k, v in ema_state.items()},
        "arch": "dense_1_4m",
        "params": n_params,
        "vocab": tok.vocab_size,
        "mix": MIX,
        "seq": SEQ,
        "batch": BATCH,
        "recipe": "crush_glint2_v2",
    }
    path = ckpt_dir / f"{tag}.pt"
    torch.save(payload, path)
    if tag == "latest":
        torch.save(payload, ckpt_dir / f"step_{step:06d}.pt")
        # Export EMA weights for generation / HF-style folder.
        backup = load_ema_into(raw_model)
        raw_model.save_pretrained(ckpt_dir / "model")
        tok.save(ckpt_dir / "model" / "tokenizer.json")
        raw_model.load_state_dict(backup)
    if drive_ckpt:
        shutil.copy2(path, drive_ckpt / path.name)
        if tag == "latest":
            shutil.copy2(ckpt_dir / f"step_{step:06d}.pt", drive_ckpt / f"step_{step:06d}.pt")
            model_drive = drive_ckpt / "model"
            if model_drive.exists():
                shutil.rmtree(model_drive)
            shutil.copytree(ckpt_dir / "model", model_drive)
            tok.save(drive_ckpt / "tokenizer.json")
            meta = {
                "step": step,
                "tokens_seen": tokens_seen,
                "loss": loss,
                "best_loss": best_loss,
                "params": n_params,
                "hours": HOURS,
                "mix": MIX,
                "seq": SEQ,
                "recipe": "crush_glint2_v2",
                "glint_targets": GLINT_TARGETS,
            }
            (drive_ckpt / "train_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"  saved {tag} @ step {step} loss={loss:.4f}")

# ---------------------------------------------------------------------------
# 6) Contiguous packed buffer + train loop (wall-clock capped)
# ---------------------------------------------------------------------------
buffer: list[int] = []


def refill(min_tokens: int = STREAM_BUFFER_TOKENS) -> None:
    while len(buffer) < min_tokens:
        ids = encode(next_doc())
        if not ids:
            continue
        buffer.extend(ids)
        buffer.append(tok.eos_id)


def sample_batch():
    """Non-overlapping packed windows — maximise unique tokens per wall-clock hour."""
    need = BATCH * (SEQ + 1)
    refill(max(STREAM_BUFFER_TOKENS, need * 2))
    xs, ys = [], []
    for _ in range(BATCH):
        if len(buffer) < SEQ + 1:
            refill(need)
        chunk = buffer[: SEQ + 1]
        del buffer[: SEQ + 1]
        xs.append(chunk[:SEQ])
        ys.append(chunk[1 : SEQ + 1])
    return (
        torch.tensor(xs, dtype=torch.long, device=device),
        torch.tensor(ys, dtype=torch.long, device=device),
    )


def soft_kl_batch(n: int = SOFT_KL_BATCH) -> torch.Tensor:
    if not aligned_soft:
        return torch.zeros((), device=device)
    total = torch.zeros((), device=device)
    usable = 0
    for i in range(n):
        step_s = aligned_soft[(start_step + step + i) % len(aligned_soft)]
        if len(step_s.context_ids) < 1:
            continue
        ctx = torch.tensor(step_s.context_ids[-SEQ:], dtype=torch.long, device=device).unsqueeze(0)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_cuda):
            logits = model(ctx)[0, -1].float()
        total = total + soft_kl_loss(logits, step_s.token_ids, step_s.probs)
        usable += 1
    return total / max(usable, 1)


model.train()
t0 = time.perf_counter()
deadline = t0 + HOURS * 3600
losses: list[float] = []
step = start_step
print(
    f"training up to {HOURS}h  ({tokens_per_step:,} tok/step, "
    f"soft KL every {SOFT_KL_EVERY}, EMA={EMA_DECAY})"
)

try:
    while time.perf_counter() < deadline:
        step += 1
        progress = (time.perf_counter() - t0) / max(HOURS * 3600, 1e-6)
        # Account for resumed runs: map remaining wall clock onto remaining schedule.
        if start_step > 0 and tokens_seen > 0:
            # Keep decay tied to *this* session's wall clock so LR still anneals.
            pass
        lr_now = wall_clock_lr(progress)
        for pg in optimizer.param_groups:
            pg["lr"] = lr_now

        optimizer.zero_grad(set_to_none=True)
        loss_acc = 0.0
        for _ in range(ACCUM):
            x, y = sample_batch()
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_cuda):
                out = model.forward_with_loss(x, y)
                loss = out.loss / ACCUM
            scaler.scale(loss).backward()
            loss_acc += float(out.loss.detach())
            tokens_seen += int(x.numel())
        if step % SOFT_KL_EVERY == 0:
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_cuda):
                skl = soft_kl_batch()
            scaler.scale(SOFT_KL_WEIGHT * skl).backward()
            loss_acc += SOFT_KL_WEIGHT * float(skl.detach())
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema_update()
        losses.append(loss_acc / ACCUM)

        if step % 50 == 0 or step == start_step + 1:
            elapsed = time.perf_counter() - t0
            avg = sum(losses[-50:]) / len(losses[-50:])
            print(
                f"step {step:>6}  loss {avg:.4f}  lr {lr_now:.2e}  "
                f"tokens {tokens_seen / 1e6:.1f}M  "
                f"{tokens_seen / max(elapsed, 1e-6):,.0f} tok/s  "
                f"eta_left {(deadline - time.perf_counter()) / 3600:.2f}h",
                flush=True,
            )
        if step % SAVE_EVERY == 0:
            avg = sum(losses[-50:]) / len(losses[-50:])
            save_ckpt(step, avg, tag="latest")
            if avg < best_loss:
                best_loss = avg
                save_ckpt(step, avg, tag="best")
except KeyboardInterrupt:
    print("interrupted — saving…")

final_loss = sum(losses[-50:]) / max(len(losses[-50:]), 1) if losses else float("nan")
if losses and final_loss < best_loss:
    best_loss = final_loss
save_ckpt(step, final_loss, tag="latest")
save_ckpt(step, final_loss, tag="best")
elapsed_h = (time.perf_counter() - t0) / 3600
print(f"DONE steps={step} tokens={tokens_seen:,} loss={final_loss:.4f} hours={elapsed_h:.2f}")

# ---------------------------------------------------------------------------
# 7) Smoke generations (EMA weights)
# ---------------------------------------------------------------------------
from minimodel.inference.sampling import generate_text

backup = load_ema_into(raw_model)
raw_model.eval()
prompts = [e["prompt"] for e in soft_entries[:8]] + [
    "Once upon a time,",
    "The capital of France is",
    "Question: What is H2O?\nAnswer:",
]
print("\n=== smoke generations (EMA) ===")
for p in prompts:
    try:
        text = generate_text(
            raw_model, tok, p, max_new_tokens=64, temperature=0.7, top_k=40, device=device
        )
    except Exception as exc:  # noqa: BLE001
        text = f"<generation failed: {exc}>"
    print(f">> {p!r}\n{text}\n")

# ---------------------------------------------------------------------------
# 8) Glint-matched eval (BLiMP / ARC-Easy / WikiText byte-ppl)
# ---------------------------------------------------------------------------
eval_metrics: dict = {}
eval_dir = repo_dir / "research" / "data" / "eval"
if (eval_dir / "blimp.jsonl").exists() and (eval_dir / "arc_easy.jsonl").exists():
    os.chdir(repo_dir)
    from eval_harness import ModelAdapter, run_all

    def _fwd(tokens: torch.Tensor) -> torch.Tensor:
        tokens = tokens.to(device)
        with torch.no_grad():
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_cuda):
                return raw_model(tokens).float().cpu()

    adapter = ModelAdapter(
        name="crush-glint2-1.4m",
        encode=lambda s: tok.encode(s, add_bos=True),
        forward=_fwd,
        max_len=min(SEQ, 512),
        batch_size=8 if use_cuda else 4,
        params=n_params,
    )
    print("\n=== Glint-matched eval (may take a few minutes) ===")
    result = run_all(adapter, blimp_per_paradigm=15, arc_limit=150, wikitext_max_tokens=10_000)
    eval_metrics = dict(result.metrics)
    print(json.dumps(eval_metrics, indent=2))
    print("Glint-2 targets:", json.dumps(GLINT_TARGETS, indent=2))
    beats = {
        "arc_easy": eval_metrics.get("arc_easy_acc", 0) >= GLINT_TARGETS["arc_easy_acc"],
        "wikitext_byte_ppl": eval_metrics.get("wikitext_byte_ppl", 1e9)
        <= GLINT_TARGETS["wikitext_byte_ppl"],
        "blimp": eval_metrics.get("blimp_acc", 0) >= GLINT_TARGETS["blimp_acc"],
        "params_smaller": n_params < GLINT_TARGETS["params"],
    }
    print("beats Glint-2?:", json.dumps(beats, indent=2))
else:
    print("eval caches missing under research/data/eval — skip harness")

raw_model.load_state_dict(backup)

summary = {
    "recipe": "crush_glint2_v2",
    "params": n_params,
    "steps": step,
    "tokens_seen": tokens_seen,
    "final_loss": final_loss,
    "best_loss": best_loss,
    "hours": elapsed_h,
    "mix": MIX,
    "lr": LR,
    "seq": SEQ,
    "batch": BATCH,
    "accum": ACCUM,
    "tokens_per_step": tokens_per_step,
    "soft_kl_every": SOFT_KL_EVERY,
    "soft_kl_weight": SOFT_KL_WEIGHT,
    "ema_decay": EMA_DECAY,
    "device": str(device),
    "eval": eval_metrics,
    "glint_targets": GLINT_TARGETS,
}
(WORK / "run_summary.json").write_text(json.dumps(summary, indent=2))
if IN_COLAB:
    shutil.copy2(WORK / "run_summary.json", DRIVE_ROOT / "run_summary.json")
print("summary:", json.dumps(summary, indent=2))
print("Drive folder:" if IN_COLAB else "Local work:", DRIVE_ROOT if IN_COLAB else WORK)
